In [ ]:
from google.colab import drive 
drive.mount('/content/drive')

In [ ]:
# Define datasets we are going to be working with, for readablity
DATASET_PATH = "/content/drive/MyDrive/Datasets/HR_128_normalized"

# extract other things for dataset name
DATASET_NAME = DATASET_PATH.split('/')[-1]
print(f"Working with dataset: {DATASET_NAME}")
ALIGNED_PATH = "/content/drive/MyDrive/Datasets_aligned"

In [6]:
# have to use those stupid fucking ! things to clone 
# and then cd cause it's not persistant and shit
!git clone https://github.com/alyssaq/face_morpher.git
%cd face_morpher

!mkdir -p data

!curl -LO https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -d shape_predictor_68_face_landmarks.dat.bz2
!mv shape_predictor_68_face_landmarks.dat data/

# and normal export is not going to work as well
import os
os.environ["DLIB_DATA_DIR"] = os.path.abspath("data")

Cloning into 'face_morpher'...
remote: Enumerating objects: 942, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 942 (delta 0), reused 1 (delta 0), pack-reused 937 (from 1)
Receiving objects: 100% (942/942), 75.59 MiB | 17.74 MiB/s, done.
Resolving deltas: 100% (575/575), done.
/content/face_morpher
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 61.0M  100 61.0M    0     0  26.7M      0  0:00:02  0:00:02 --:--:-- 26.8M


In [7]:
%ls

data/      facemorpher/  requirements.txt  setup.cfg
examples/  README.rst    scripts/          setup.py


In [8]:
%%bash
# installing the stuff and settng up the morpher as module
pip install -r requirements.txt
pip install -e .

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13781 sha256=6da439e0f8a7ea93219e04a255b66cf57be78120669908843105b57d7d223981
  Stored in directory: /root/.cache/pip/wheels/0b/1d/03/175286677fb5a1341cc3e4755bf8ec0ed08f3329afd67446b0
Successfully built docopt
Obtaining file:///content/face_morpher
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py develop for facemorpher


In [ ]:
# python imports block
import subprocess
import sys
from facemorpher import morpher
import os
import glob
import bz2
import requests
import numpy as np
import scipy.ndimage
import PIL.Image
import dlib
from tqdm import tqdm

In [ ]:
# alignment functions
# --- FFHQ Standard Image Alignment Transform ---
def image_align(src_file, dst_file, face_landmarks, output_size=256, transform_size=4096, enable_padding=True):
    lm = np.array(face_landmarks)
    lm_eye_left = lm[36:42]
    lm_eye_right = lm[42:48]
    lm_mouth_outer = lm[48:60]

    eye_left = np.mean(lm_eye_left, axis=0)
    eye_right = np.mean(lm_eye_right, axis=0)
    eye_avg = (eye_left + eye_right) * 0.5
    eye_to_eye = eye_right - eye_left
    mouth_left = lm_mouth_outer[0]
    mouth_right = lm_mouth_outer[6]
    mouth_avg = (mouth_left + mouth_right) * 0.5
    eye_to_mouth = mouth_avg - eye_avg

    x = eye_to_eye - np.flipud(eye_to_mouth) * [-1, 1]
    x /= np.hypot(*x)
    x *= max(np.hypot(*eye_to_eye) * 2.0, np.hypot(*eye_to_mouth) * 1.8)
    y = np.flipud(x) * [-1, 1]
    c = eye_avg + eye_to_mouth * 0.1
    quad = np.stack([c - x - y, c - x + y, c + x + y, c + x - y])
    qsize = np.hypot(*x) * 2

    img = PIL.Image.open(src_file).convert('RGB')

    shrink = int(np.floor(qsize / output_size * 0.5))
    if shrink > 1:
        rsize = (int(np.rint(float(img.size[0]) / shrink)), int(np.rint(float(img.size[1]) / shrink)))
        img = img.resize(rsize, PIL.Image.LANCZOS)
        quad /= shrink
        qsize /= shrink

    border = max(int(np.rint(qsize * 0.1)), 3)
    crop = (int(np.floor(min(quad[:, 0]))), int(np.floor(min(quad[:, 1]))),
            int(np.ceil(max(quad[:, 0]))), int(np.ceil(max(quad[:, 1]))))
    crop = (max(crop[0] - border, 0), max(crop[1] - border, 0),
            min(crop[2] + border, img.size[0]), min(crop[3] + border, img.size[1]))
    if crop[2] - crop[0] < img.size[0] or crop[3] - crop[1] < img.size[1]:
        img = img.crop(crop)
        quad -= crop[0:2]

    pad = (int(np.floor(min(quad[:, 0]))), int(np.floor(min(quad[:, 1]))),
           int(np.ceil(max(quad[:, 0]))), int(np.ceil(max(quad[:, 1]))))
    pad = (max(-pad[0] + border, 0), max(-pad[1] + border, 0),
           max(pad[2] - img.size[0] + border, 0), max(pad[3] - img.size[1] + border, 0))
    if enable_padding and max(pad) > border - 4:
        pad = np.maximum(pad, int(np.rint(qsize * 0.3)))
        img = np.pad(np.float32(img), ((pad[1], pad[3]), (pad[0], pad[2]), (0, 0)), 'reflect')
        h, w, _ = img.shape
        y_grid, x_grid, _ = np.ogrid[:h, :w, :1]
        mask = np.maximum(
            1.0 - np.minimum(np.float32(x_grid) / pad[0], np.float32(w - 1 - x_grid) / pad[2]),
            1.0 - np.minimum(np.float32(y_grid) / pad[1], np.float32(h - 1 - y_grid) / pad[3])
        )
        blur = qsize * 0.02
        img += (scipy.ndimage.gaussian_filter(img, [blur, blur, 0]) - img) * np.clip(mask * 3.0 + 1.0, 0.0, 1.0)
        img += (np.median(img, axis=(0, 1)) - img) * np.clip(mask, 0.0, 1.0)
        img = PIL.Image.fromarray(np.uint8(np.clip(np.rint(img), 0, 255)), 'RGB')
        quad += pad[:2]

    img = img.transform((transform_size, transform_size), PIL.Image.QUAD, (quad + 0.5).flatten(), PIL.Image.BILINEAR)
    if output_size < transform_size:
        img = img.resize((output_size, output_size), PIL.Image.LANCZOS)

    img.save(dst_file, 'PNG')

# --- Recursive Dataset Pipeline ---
def align_dataset(input_root, output_root):
    model_path = get_shape_predictor()
    detector = dlib.get_frontal_face_detector()
    predictor = dlib.shape_predictor(model_path)

    # Recursively gather images across all subfolders
    extensions = ('*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG')
    image_paths = []
    for ext in extensions:
        image_paths.extend(glob.glob(os.path.join(input_root, '**', ext), recursive=True))

    print(f"Discovered {len(image_paths)} images across dataset tree.")

    for img_path in tqdm(image_paths, desc="Aligning Faces"):
        rel_path = os.path.relpath(img_path, input_root)
        name_without_ext = os.path.splitext(rel_path)[0]
        out_path = os.path.join(output_root, f"{name_without_ext}.png")

        if os.path.exists(out_path):
            continue

        os.makedirs(os.path.dirname(out_path), exist_ok=True)

        try:
            img = dlib.load_rgb_image(img_path)
            dets = detector(img, 1)
            if len(dets) == 0:
                continue

            # Align using primary detected face
            landmarks = [(p.x, p.y) for p in predictor(img, dets[0]).parts()]
            image_align(img_path, out_path, landmarks, output_size=256)
        except Exception as e:
            print(f"Skipping {img_path}: {e}")

    print(f"Alignment completed. Files saved to: {output_root}")

In [ ]:
# aligning the dataset if it hasn't been done before
if not os.path.exists(f"{ALIGNED_PATH}/{DATASET_NAME}"):
    align_dataset(
        input_root=DATASET_PATH, 
        output_root=f"{ALIGNED_PATH}/{DATASET_NAME}"
    )
else: 
    print("Aligned path exists, skipping the alignment process")